# 🌍 Project 23 — Mosaic Multiple Images

## 🎯 Goal

Combine neighboring satellite scenes into one seamless image.

Instead of displaying separate satellite tiles, we'll merge them into a single image.

---

# 🌎 Why is this useful in Remote Sensing?

Mosaics are widely used for:

- 🗺 Regional mapping
- 🌍 Country-wide satellite maps
- 🌎 Continental mapping
- 🚜 Agriculture
- 🌳 Forest monitoring

Many study areas are larger than a single satellite scene.

A mosaic allows us to analyze the entire area as one image.

---

# 🆕 New Skill: Merging Images

Previously:

```text
Many Images
      │
      ▼
   median()
      │
      ▼
 Cleaner Image
```

Today:

```text
Image A
      +

Image B
      +

Image C
      │
      ▼
   mosaic()
      │
      ▼
One Seamless Image
```

Notice that `mosaic()` **does not calculate averages or medians**.

It simply joins images together.

---

# 🔗 Connect to Previous Lessons

| Project | Skill |
|----------|-------|
| Project 3 | ImageCollection |
| Project 18 | `median()` |
| Project 22 | Median Composite |
| Project 23 | `mosaic()` |

Today you'll learn **when to use `mosaic()` instead of `median()`**.

---

# 📖 What is a Mosaic?

Imagine Germany is covered by three satellite scenes.

```text
Scene 1

██████


Scene 2

      ██████


Scene 3

            ██████
```

Instead of displaying them separately:

```text
Scene 1

Scene 2

Scene 3
```

Earth Engine joins them into:

```text
████████████████████
```

One seamless image.

---

# ⭐ Median vs Mosaic

This is the most important concept of today's lesson.

## `median()`

```text
Same Location

Image 1

Image 2

Image 3
      │
      ▼
 Median Pixel
```

**Purpose:**

➡️ Reduce clouds and noise.

---

## `mosaic()`

```text
Image A

██████

Image B

      ██████

Image C

            ██████

      │
      ▼

████████████████████
```

**Purpose:**

➡️ Merge neighboring images.

---

# Step 1 — Define the Study Area

We'll use Germany.

```javascript
var countryArea = ee.Geometry.Polygon([
  [
    [5.87,47.27],
    [15.04,47.27],
    [15.04,55.06],
    [5.87,55.06],
    [5.87,47.27]
  ]
]);
```

---

# Step 2 — Load the Sentinel-2 Collection

```javascript
var collection = ee.ImageCollection(
  "COPERNICUS/S2_SR_HARMONIZED"
)
.filterBounds(countryArea)
.filterDate('2024-06-01','2024-06-10');
```

Notice:

We intentionally use a **short date range**.

**Why?**

Because several Sentinel-2 scenes may cover Germany during the same period.

---

# ⭐ Step 3 — Create the Mosaic

Today's new function:

```javascript
var mosaicImage = collection.mosaic();
```

This is the key step.

### Input

```text
Image Collection
├── Image 1
├── Image 2
├── Image 3
```

### Output

```text
One Image
```

The output type is:

```text
ee.Image
```

---

# 🤔 How Does `mosaic()` Choose Pixels?

This is an important difference from `median()`.

Suppose two images overlap.

```text
Image A
████████

Image B
    ████████
```

Earth Engine starts with the **last image** in the collection.

If a pixel exists there, it is used.

If not, it looks at the previous image, and so on.

Workflow:

```text
Last Image
      │
      ▼
Previous Image
      │
      ▼
Previous Image
      │
      ▼
...
```

So **image order matters**.

---

# Step 4 — Clip the Mosaic

```javascript
mosaicImage = mosaicImage.clip(countryArea);
```

---

# Step 5 — Visualization

```javascript
var trueColor = {
  bands:['B4','B3','B2'],
  min:0,
  max:3000
};
```

---

# Step 6 — Display

```javascript
Map.centerObject(countryArea,5);

Map.addLayer(
  mosaicImage,
  trueColor,
  'Mosaic Image'
);
```

The result is one seamless image covering the study area.

---

# ⭐ Bonus — Change Image Order

Since `mosaic()` depends on image order, try this:

```javascript
var sortedCollection = collection.sort(
  'CLOUDY_PIXEL_PERCENTAGE'
);

var mosaicImage = sortedCollection.mosaic();
```

Now the images are ordered by cloud cover before creating the mosaic.

Turn the layers on and off and see whether the mosaic changes.

---

# 💻 Complete Code

```javascript
// =====================================
// 1. Study Area
// =====================================
var countryArea = ee.Geometry.Polygon([
  [
    [5.87,47.27],
    [15.04,47.27],
    [15.04,55.06],
    [5.87,55.06],
    [5.87,47.27]
  ]
]);

// =====================================
// 2. Load Collection
// =====================================
var collection = ee.ImageCollection(
  "COPERNICUS/S2_SR_HARMONIZED"
)
.filterBounds(countryArea)
.filterDate('2024-06-01','2024-06-10');

// =====================================
// 3. Create Mosaic
// =====================================
var mosaicImage = collection
  .mosaic()
  .clip(countryArea);

// =====================================
// 4. Visualization
// =====================================
var trueColor = {
  bands:['B4','B3','B2'],
  min:0,
  max:3000
};

// =====================================
// 5. Display
// =====================================
Map.centerObject(countryArea,5);

Map.addLayer(
  mosaicImage,
  trueColor,
  'Mosaic Image'
);
```

---

# 📝 Things to Remember

## 1. `mosaic()`

`mosaic()` combines an **ImageCollection** into a single **ee.Image** by **merging images together**.

```text
Image Collection
       │
       ▼
    mosaic()
       │
       ▼
    One Image
```

---

## 2. Image Order Matters

Unlike `median()`, `mosaic()` does **not** calculate new pixel values.

It simply selects pixels from the images based on their order in the collection.

```text
Last Image
      │
      ▼
Previous Image
      │
      ▼
Previous Image
```

This is why **sorting the collection before using `mosaic()`** can affect the final result.

---

## 3. `median()` vs `mosaic()`

| `median()` | `mosaic()` |
|------------|------------|
| Calculates the median pixel value | Copies pixels from existing images |
| Reduces clouds and noise | Merges neighboring or overlapping scenes |
| Image order does not matter | Image order matters |
| Creates a statistical composite | Creates a seamless image |

Knowing when to use each function is an important intermediate GEE skill.

---

# ⚠ Common Beginner Mistakes

### ❌ Mistake 1: Thinking `mosaic()` removes clouds

It does **not** calculate better pixel values.

If the top image contains clouds, those clouds may appear in the mosaic.

---

### ❌ Mistake 2: Forgetting that image order matters

If two images overlap, the **last image** in the collection has priority.

---

### ❌ Mistake 3: Using `mosaic()` when you really need `median()`

If your goal is **cloud reduction over time**, use:

```javascript
median()
```

If your goal is **to merge neighboring scenes into one continuous image**, use:

```javascript
mosaic()
```

---

# 💻 Your Practice (Project 23.1)

Create a **mosaic image** for a large study area.

### Requirements

- Create a study area.
- Load the Sentinel-2 ImageCollection.
- Filter by location and a short date range.
- Create a mosaic using `.mosaic()`.
- Clip the result.
- Display the mosaic.
- **Bonus:** Sort the collection by `CLOUDY_PIXEL_PERCENTAGE` before creating the mosaic and compare the results.

---

# ❓ Before Writing the Code

## Question 1

Both of these functions return an `ee.Image`:

```javascript
collection.median()
```

and

```javascript
collection.mosaic()
```

What is the main difference between **how they create that image**?

---

## Question 2

Why can changing the order of images in an ImageCollection change the result of:

```javascript
collection.mosaic()
```

but usually not change the result of:

```javascript
collection.median()
```